# XGBoost Forecasting — Training & Verification

This notebook walks through the full pipeline:

1. **Data loading** — read live telemetry CSVs
2. **Preprocessing** — resample to 5s buckets
3. **Feature engineering** — lag feature matrix (lag-only, no time features)
4. **Training** — fit XGBRegressor per signal
5. **Model inspection** — feature importances, prediction vs actual
6. **Residual analysis** — z-score distribution, anomaly threshold
7. **Anomaly injection tests** — inject artificial attacks, verify detection
8. **Inference simulation** — step through the live deque-based inference loop
9. **All-signals test** — verify all 7 loaded models produce sensible predictions

Run from `backend/` directory, or adjust `BACKEND_DIR` below.

In [ ]:
import sys
from pathlib import Path

# Ensure backend/ is on sys.path so forecasting.* imports work
BACKEND_DIR = Path(".").resolve()
if not (BACKEND_DIR / "forecasting").exists():
    BACKEND_DIR = BACKEND_DIR.parent
if not (BACKEND_DIR / "forecasting").exists():
    BACKEND_DIR = BACKEND_DIR.parent

if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

print(f"Backend dir      : {BACKEND_DIR}")
print(f"forecasting/ OK  : {(BACKEND_DIR / 'forecasting').exists()}")

In [ ]:
from collections import deque
import json
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from xgboost import XGBRegressor

from forecasting.config import (
    ANOMALY_Z_THRESHOLD,
    LAG_WINDOW,
    LOG_DIR,
    MIN_TRAINING_BUCKETS,
    MODELS_DIR,
    RESIDUAL_STATS_PATH,
    SIGNALS,
    SUSTAINED_ANOMALY_THRESHOLD,
    XGB_LEARNING_RATE,
    XGB_MAX_DEPTH,
    XGB_N_ESTIMATORS,
)
from forecasting.preprocess import load_sensor_logs, resample_to_5s

plt.rcParams["figure.figsize"] = (14, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print(f"LAG_WINDOW        = {LAG_WINDOW}")
print(f"XGB_N_ESTIMATORS  = {XGB_N_ESTIMATORS}")
print(f"XGB_MAX_DEPTH     = {XGB_MAX_DEPTH}")
print(f"XGB_LEARNING_RATE = {XGB_LEARNING_RATE}")
print(f"ANOMALY_Z_THRESH  = {ANOMALY_Z_THRESHOLD}σ")
print(f"SUSTAINED_THRESH  = {SUSTAINED_ANOMALY_THRESHOLD} consecutive buckets")
print(f"LOG_DIR           = {LOG_DIR}")
print()
print("Features: [v(t-1), v(t-2), v(t-3), v(t-4), v(t-5)]  — lag-only, no time-of-day")

---
## 1. Data Loading

In [ ]:
DEMO_SIGNAL = "load1_ac_voltage"
cfg = SIGNALS[DEMO_SIGNAL]

raw = load_sensor_logs(cfg["sensor"], cfg["field"], LOG_DIR)

print(f"Signal  : {DEMO_SIGNAL}")
print(f"Sensor  : {cfg['sensor']}  |  Field: {cfg['field']}")
print(f"Rows    : {len(raw):,}")
if not raw.empty:
    print(f"Range   : {raw.index[0]}  →  {raw.index[-1]}")
    print(f"Mean    : {raw.mean():.3f}")
    print(f"Std     : {raw.std():.3f}")
    print(f"Min/Max : {raw.min():.3f} / {raw.max():.3f}")
raw.head(8)

In [ ]:
fig, ax = plt.subplots()
raw.plot(ax=ax, linewidth=0.6, alpha=0.8, color="steelblue")
ax.set_title(f"{DEMO_SIGNAL} — raw readings (every 0.5 s)")
ax.set_ylabel(cfg["field"])
ax.set_xlabel("timestamp (UTC)")
plt.tight_layout()
plt.show()

---
## 2. Preprocessing — Resample to 5 s Buckets

In [ ]:
resampled = resample_to_5s(raw)

compression = len(raw) / len(resampled) if not resampled.empty else 0
status = "OK" if len(resampled) >= MIN_TRAINING_BUCKETS else "NOT ENOUGH — run mock_training_data.py"

print(f"Raw rows      : {len(raw):,}")
print(f"Resampled rows: {len(resampled):,}  ({compression:.1f}× compression)")
print(f"Need ≥ {MIN_TRAINING_BUCKETS} buckets: {status}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

raw.iloc[:600].plot(ax=axes[0], linewidth=0.5, color="steelblue", alpha=0.7)
axes[0].set_title("Raw readings — first 5 minutes")

resampled.iloc[:60].plot(ax=axes[1], linewidth=1.2, color="darkorange", marker="o", markersize=3)
axes[1].set_title("Resampled to 5 s — first 5 minutes")

plt.tight_layout()
plt.show()

---
## 3. Feature Engineering — Lag Matrix

| Feature | Meaning |
|---------|--------|
| `lag_1` | value 5 s ago — v(t-1) |
| `lag_2` | value 10 s ago — v(t-2) |
| `lag_3` | value 15 s ago — v(t-3) |
| `lag_4` | value 20 s ago — v(t-4) |
| `lag_5` | value 25 s ago — v(t-5) |
| **target** | **current value — what we predict** |

> **Why no hour/minute?** For a 5 s prediction horizon, the recent lag values carry all necessary
> context. Time-of-day features generalise poorly when training data covers only a few hours of the
> day — they would shift predictions wildly at unseen times, causing flood-anomalies like z=1500.

In [ ]:
def build_features(series: pd.Series, lag: int):
    """Build (X, y) arrays — identical to train.py."""
    vals = series.values
    X, y = [], []
    for i in range(lag, len(vals)):
        lags = vals[i - lag:i][::-1]   # [t-1, t-2, ..., t-lag]
        X.append(lags)
        y.append(vals[i])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X, y = build_features(resampled, LAG_WINDOW)
feature_names = [f"lag_{i+1}" for i in range(LAG_WINDOW)]

print(f"X shape: {X.shape}  ({X.shape[0]} training samples, {X.shape[1]} features)")
print(f"y shape: {y.shape}")

df_sample = pd.DataFrame(X[:6], columns=feature_names)
df_sample["→ target"] = y[:6]
df_sample.round(4)

In [ ]:
# Sanity check: lag_1 for row i must equal target for row i-1
assert abs(X[1, 0] - y[0]) < 1e-4, "lag_1 mismatch!"
print("Lag consistency check PASSED")
print(f"  y[0]        = {y[0]:.4f}  (target at bucket 0)")
print(f"  X[1, lag_1] = {X[1, 0]:.4f}  (should be identical — it's 5 s earlier)")

---
## 4. Training

In [ ]:
model = XGBRegressor(
    n_estimators=XGB_N_ESTIMATORS,
    max_depth=XGB_MAX_DEPTH,
    learning_rate=XGB_LEARNING_RATE,
    tree_method="hist",
    verbosity=0,
)
model.fit(X, y)
print("Training complete.")

preds = model.predict(X)
residuals = y - preds
res_mean = float(residuals.mean())
res_std  = float(residuals.std())

print(f"\nIn-sample metrics")
print(f"  MAE              : {np.abs(residuals).mean():.4f}")
print(f"  RMSE             : {np.sqrt((residuals**2).mean()):.4f}")
print(f"  Residual mean    : {res_mean:.6f}")
print(f"  Residual std     : {res_std:.4f}")
print(f"  Anomaly threshold: ±{ANOMALY_Z_THRESHOLD * res_std:.4f} units  (={ANOMALY_Z_THRESHOLD}σ)")

---
## 5. Model Inspection

In [ ]:
importances = model.feature_importances_

fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.barh(feature_names, importances, color="steelblue")
ax.set_xlabel("Feature importance (gain)")
ax.set_title(f"{DEMO_SIGNAL} — XGBoost feature importances")
ax.invert_yaxis()
for bar, val in zip(bars, importances):
    ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

print("Expected: lag_1 dominates — the most recent value is the best predictor of the next.")
print(f"lag_1 importance: {importances[0]:.3f}")

In [ ]:
WINDOW = 200
ts_idx   = resampled.index[LAG_WINDOW:]
ts_w     = ts_idx[-WINDOW:]
y_w      = y[-WINDOW:]
p_w      = preds[-WINDOW:]
res_w    = y_w - p_w

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(ts_w, y_w, label="Actual",    linewidth=1.2, color="#22c55e")
axes[0].plot(ts_w, p_w, label="Predicted", linewidth=1.0, color="#3b82f6", linestyle="--", alpha=0.85)
axes[0].set_title(f"{DEMO_SIGNAL} — predicted vs actual (last {WINDOW} buckets)")
axes[0].set_ylabel("value")
axes[0].legend()

axes[1].plot(ts_w, res_w, linewidth=0.8, color="gray", label="Residual")
axes[1].axhline(res_mean + ANOMALY_Z_THRESHOLD * res_std, color="red", linestyle="--", linewidth=1, label=f"+{ANOMALY_Z_THRESHOLD}σ")
axes[1].axhline(res_mean - ANOMALY_Z_THRESHOLD * res_std, color="red", linestyle="--", linewidth=1)
axes[1].axhline(0, color="black", linewidth=0.5, alpha=0.4)
axes[1].set_ylabel("residual")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 6. Residual Analysis

In [ ]:
z_scores = (residuals - res_mean) / res_std

n_anomaly = int(np.sum(np.abs(z_scores) > ANOMALY_Z_THRESHOLD))
pct = 100 * n_anomaly / len(z_scores)

print("Z-score distribution")
print(f"  Mean : {z_scores.mean():.4f}  (should be ≈ 0)")
print(f"  Std  : {z_scores.std():.4f}   (should be ≈ 1)")
print(f"  Min  : {z_scores.min():.3f}")
print(f"  Max  : {z_scores.max():.3f}")
print(f"\nIn-sample |z| > {ANOMALY_Z_THRESHOLD}: {n_anomaly}/{len(z_scores)} ({pct:.2f}%)")
print("Normal distribution expect ≈ 0.27% at 3σ.")

In [ ]:
from scipy import stats as scipy_stats

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(z_scores, bins=60, color="steelblue", alpha=0.7, edgecolor="white", linewidth=0.3)
axes[0].axvline( ANOMALY_Z_THRESHOLD, color="red", linestyle="--", label=f"+{ANOMALY_Z_THRESHOLD}σ")
axes[0].axvline(-ANOMALY_Z_THRESHOLD, color="red", linestyle="--")
axes[0].set_xlabel("z-score")
axes[0].set_ylabel("count")
axes[0].set_title("In-sample z-score distribution")
axes[0].legend()

(osm, osr), (slope, intercept, r) = scipy_stats.probplot(z_scores, dist="norm")
axes[1].plot(osm, osr, "o", markersize=2, alpha=0.5, color="steelblue")
axes[1].plot(osm, slope * np.array(osm) + intercept, "r-", linewidth=1.5, label=f"Normal fit  R={r:.3f}")
axes[1].set_xlabel("theoretical quantiles")
axes[1].set_ylabel("sample quantiles")
axes[1].set_title("Q-Q plot")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 7. Anomaly Injection Tests

In [ ]:
def simulate_inference(series_slice: pd.Series, attack_start: int, attack_end: int, attack_delta: float):
    """Step through series_slice bucket-by-bucket with the same logic as state.process_bucket()."""
    history = deque(maxlen=LAG_WINDOW)
    consecutive_anomalies = 0
    rows = []

    for i, (ts_raw, real_val) in enumerate(series_slice.items()):
        actual = float(real_val) + (attack_delta if attack_start <= i < attack_end else 0.0)
        attacked = attack_start <= i < attack_end

        if len(history) < LAG_WINDOW:
            history.append(actual)
            rows.append({"i": i, "actual": actual, "predicted": None, "z_score": None,
                         "is_anomaly": False, "attacked": attacked, "consecutive": 0,
                         "sustained": False, "status": "warming_up"})
            continue

        lags = list(reversed(history))
        features = np.array([lags], dtype=np.float32)
        predicted = float(model.predict(features)[0])
        residual = actual - predicted
        z_score = (residual - res_mean) / res_std if res_std > 1e-9 else 0.0
        is_anomaly = abs(z_score) > ANOMALY_Z_THRESHOLD

        if is_anomaly:
            history.append(predicted)   # impute
            consecutive_anomalies += 1
        else:
            history.append(actual)
            consecutive_anomalies = 0

        sustained = consecutive_anomalies >= SUSTAINED_ANOMALY_THRESHOLD
        rows.append({"i": i, "actual": actual, "predicted": predicted, "z_score": z_score,
                     "is_anomaly": is_anomaly, "attacked": attacked, "consecutive": consecutive_anomalies,
                     "sustained": sustained, "status": "anomaly" if is_anomaly else "normal"})

    return pd.DataFrame(rows).set_index("i")

print("simulate_inference() ready.")

### Test 1: Spike attack (+20V for 15 buckets)

In [ ]:
test_slice   = resampled.iloc[-120:]
ATTACK_START = 40
ATTACK_END   = 55
ATTACK_DELTA = 20.0

r1 = simulate_inference(test_slice, ATTACK_START, ATTACK_END, ATTACK_DELTA)

print("── Test 1: +20 unit spike ──")
print(f"Attack window     : buckets {ATTACK_START}–{ATTACK_END-1}")
print(f"Anomalies flagged : {r1['is_anomaly'].sum()}")
print(f"Detection rate    : {r1.loc[r1['attacked'], 'is_anomaly'].mean():.0%} of attacked buckets")
print(f"False positive rate: {r1.loc[~r1['attacked'] & r1['z_score'].notna(), 'is_anomaly'].mean():.2%}")
print(f"Sustained flag    : {r1['sustained'].any()}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

ready = r1[r1["predicted"].notna()]
axes[0].plot(ready.index, ready["actual"],    label="Actual",    color="#22c55e", linewidth=1.2)
axes[0].plot(ready.index, ready["predicted"], label="Predicted", color="#3b82f6", linewidth=1.0, linestyle="--")
axes[0].axvspan(ATTACK_START, ATTACK_END - 1, alpha=0.15, color="red", label="Attack window")
det = r1[r1["is_anomaly"]]
axes[0].scatter(det.index, det["actual"], color="red", zorder=5, s=30, label="Detected")
axes[0].set_ylabel("value")
axes[0].set_title(f"Test 1: +{ATTACK_DELTA} spike on {DEMO_SIGNAL}")
axes[0].legend()

rz = r1[r1["z_score"].notna()]
axes[1].plot(rz.index, rz["z_score"], color="gray", linewidth=0.9, label="Z-score")
axes[1].axhline( ANOMALY_Z_THRESHOLD, color="red", linestyle="--", linewidth=1, label=f"±{ANOMALY_Z_THRESHOLD}σ")
axes[1].axhline(-ANOMALY_Z_THRESHOLD, color="red", linestyle="--", linewidth=1)
axes[1].axhline(0, color="black", linewidth=0.4, alpha=0.4)
axes[1].axvspan(ATTACK_START, ATTACK_END - 1, alpha=0.15, color="red")
axes[1].set_ylabel("z-score")
axes[1].set_xlabel("bucket index")
axes[1].legend()

plt.tight_layout()
plt.show()

### Test 2: Sustained attack → sustained flag fires after 10 consecutive

In [ ]:
r2 = simulate_inference(test_slice, ATTACK_START, ATTACK_START + 30, ATTACK_DELTA)

print("── Test 2: 30-bucket sustained attack ──")
print(f"Anomalies flagged : {r2['is_anomaly'].sum()}")
print(f"Sustained flag    : {r2['sustained'].any()}")
if r2["sustained"].any():
    first = r2[r2["sustained"]].index[0]
    print(f"First sustained at bucket {first}  (expected ≈ {ATTACK_START + SUSTAINED_ANOMALY_THRESHOLD - 1})")

### Test 3: Sub-threshold noise — should NOT trigger

In [ ]:
small_delta = 0.5 * ANOMALY_Z_THRESHOLD * res_std
r3 = simulate_inference(test_slice, ATTACK_START, ATTACK_END, small_delta)

print("── Test 3: Sub-threshold noise ──")
print(f"Attack delta     : {small_delta:.4f}  ({0.5 * ANOMALY_Z_THRESHOLD:.1f}σ)")
print(f"Anomalies flagged: {r3['is_anomaly'].sum()}  (expect 0)")
window_z = r3.loc[ATTACK_START:ATTACK_END - 1, "z_score"]
print(f"Max |z| in window: {window_z.abs().max():.3f}  (threshold={ANOMALY_Z_THRESHOLD})")

### Test 4: Imputation — model stays accurate after attack ends

In [ ]:
def simulate_no_imputation(series_slice, attack_start, attack_end, attack_delta):
    history = deque(maxlen=LAG_WINDOW)
    rows = []
    for i, (_, real_val) in enumerate(series_slice.items()):
        actual = float(real_val) + (attack_delta if attack_start <= i < attack_end else 0.0)
        if len(history) < LAG_WINDOW:
            history.append(actual)
            rows.append({"i": i, "predicted": None})
            continue
        lags = list(reversed(history))
        features = np.array([lags], dtype=np.float32)
        predicted = float(model.predict(features)[0])
        history.append(actual)   # no imputation — attack poisons history
        rows.append({"i": i, "predicted": predicted})
    return pd.DataFrame(rows).set_index("i")

r_imp  = simulate_inference(test_slice, ATTACK_START, ATTACK_END, ATTACK_DELTA)
r_noi  = simulate_no_imputation(test_slice, ATTACK_START, ATTACK_END, ATTACK_DELTA)

actual_post = test_slice.iloc[ATTACK_END:ATTACK_END + 20].values
pred_imp  = r_imp.loc[ATTACK_END:ATTACK_END + 19, "predicted"].values.astype(float)
pred_noi  = r_noi.loc[ATTACK_END:ATTACK_END + 19, "predicted"].values.astype(float)

mae_imp = np.abs(actual_post - pred_imp).mean()
mae_noi = np.abs(actual_post - pred_noi).mean()

print("── Test 4: Post-attack MAE ──")
print(f"WITH imputation   : {mae_imp:.4f}")
print(f"WITHOUT imputation: {mae_noi:.4f}")
improvement = (mae_noi - mae_imp) / mae_noi * 100 if mae_noi > 0 else 0
print(f"Improvement       : {improvement:.1f}% better with imputation")

---
## 8. Inference Simulation — Bucket-by-Bucket

Replicate exactly what `inference.py` does at runtime.

In [ ]:
print(f"{'#':>5} {'timestamp':>28} {'actual':>10} {'predicted':>10} {'z':>7}  status")
print("-" * 75)

history = deque(maxlen=LAG_WINDOW)
for i, (ts_raw, val) in enumerate(resampled.iloc[-30:].items()):
    actual = float(val)

    if len(history) < LAG_WINDOW:
        history.append(actual)
        print(f"{i:>5} {str(ts_raw)[:28]:>28} {actual:>10.4f} {'—':>10} {'—':>7}  warming up ({len(history)}/{LAG_WINDOW})")
        continue

    lags = list(reversed(history))
    features = np.array([lags], dtype=np.float32)
    predicted = float(model.predict(features)[0])
    residual = actual - predicted
    z = (residual - res_mean) / res_std
    is_anomaly = abs(z) > ANOMALY_Z_THRESHOLD
    history.append(predicted if is_anomaly else actual)

    status = "ANOMALY ←" if is_anomaly else "normal"
    print(f"{i:>5} {str(ts_raw)[:28]:>28} {actual:>10.4f} {predicted:>10.4f} {z:>7.3f}  {status}")

---
## 9. All-Signals Test

Load every saved pkl, feed it the last 50 resampled buckets, verify predictions are in the correct range.

In [ ]:
if not RESIDUAL_STATS_PATH.exists():
    print("residual_stats.json missing — run: python forecasting/train.py --all")
else:
    with open(RESIDUAL_STATS_PATH) as f:
        all_stats = json.load(f)

    print(f"{'Signal':<25} {'N':>6} {'ActMean':>9} {'PredMean':>9} {'MAE':>7} {'Thresh':>8}  status")
    print("-" * 85)

    all_ok = True
    for sig_name, sig_cfg in SIGNALS.items():
        model_path = MODELS_DIR / f"{sig_name}.pkl"
        if not model_path.exists():
            print(f"{sig_name:<25}  MISSING pkl")
            all_ok = False
            continue
        if sig_name not in all_stats:
            print(f"{sig_name:<25}  MISSING stats")
            all_ok = False
            continue

        with open(model_path, "rb") as f:
            m = pickle.load(f)

        raw_s = load_sensor_logs(sig_cfg["sensor"], sig_cfg["field"], LOG_DIR)
        rs = resample_to_5s(raw_s)

        if len(rs) < LAG_WINDOW + 5:
            print(f"{sig_name:<25}  not enough data")
            all_ok = False
            continue

        X_t, y_t = build_features(rs.iloc[-(50 + LAG_WINDOW):], LAG_WINDOW)
        p_t = m.predict(X_t)
        mae = float(np.abs(y_t - p_t).mean())
        thresh = ANOMALY_Z_THRESHOLD * all_stats[sig_name]["std"]
        sane = mae < thresh * 2

        print(f"{sig_name:<25} {len(X_t):>6} {y_t.mean():>9.3f} {p_t.mean():>9.3f} {mae:>7.4f} {thresh:>8.4f}  {'OK' if sane else 'WARN — large MAE'}")

    print()
    print("All models OK." if all_ok else "Some models missing — run train.py --all")

---
## 10. Manual Prediction — Feed Any Values You Want

In [ ]:
# ── Edit these to test any scenario ──────────────────────────────────────────
# Five most-recent values for load1_ac_voltage (most recent first)
manual_lags = [230.1, 230.3, 230.0, 229.9, 230.2]
# ─────────────────────────────────────────────────────────────────────────────

feat = np.array([manual_lags], dtype=np.float32)
pred_val = float(model.predict(feat)[0])

print(f"Lag features (t-1 → t-5) : {manual_lags}")
print(f"Predicted next value     : {pred_val:.4f}")
print()

# Simulate an attack: what if the actual coming in is +20V over prediction?
for delta in [0, 5, 10, 20, 50]:
    injected = pred_val + delta
    res = injected - pred_val
    z = (res - res_mean) / res_std
    flag = "ANOMALY" if abs(z) > ANOMALY_Z_THRESHOLD else "normal"
    print(f"  actual={injected:>8.3f}  residual={res:>7.3f}  z={z:>8.3f}  → {flag}")